# 🎫 Semana 6 · Unidad 2 — Filas (Queues)

## Información del Curso

| Aspecto | Detalle |
|--------|--------|
| **Universidad** | Universidad de Talca, Chile |
| **Carrera** | Ingeniería Civil en Informática |
| **Semestre** | 2°-3° año |
| **Curso** | Algoritmos y Estructuras de Datos |
| **Docente** | PhD. César Astudillo |
| **Clase** | Semana 6 · Unidad 2 — Filas (FIFO) |
| **Duración estimada** | 90 minutos |
| **Prerequisitos** | Pilas (Stacks - LIFO), Nodos y Listas Enlazadas |

## Mapa de la Clase

```
  1. TDA Fila (15 min)
       ↓
  2. Implementación con Deque (20 min)
       ↓
  3. Implementación con Lista Enlazada (20 min)
       ↓
  4. Aplicaciones Clásicas (25 min)
       ↓
  5. Análisis de Complejidad (10 min)
       ↓
  6. Widget Interactivo + Ejercicios (10 min)
```

## Verificación de Dependencias

In [ ]:
import sys
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np
from collections import deque
import time
from timeit import timeit
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Verificar que filas.py está disponible
try:
    from filas import FilaTDA, FilaDeque, FilaEnlazada, FilaArreglo
    print("✓ Módulo 'filas' importado correctamente")
    print(f"  - FilaTDA: Clase abstracta")
    print(f"  - FilaDeque: Implementación con collections.deque")
    print(f"  - FilaEnlazada: Implementación con lista enlazada (dos punteros)")
    print(f"  - FilaArreglo: Alias de FilaDeque")
except ImportError as e:
    print(f"✗ Error al importar 'filas': {e}")
    print("  Asegúrate de que filas.py está en el mismo directorio o en PYTHONPATH")

print(f"\n✓ Matplotlib versión: {plt.matplotlib.__version__}")
print(f"✓ NumPy versión: {np.__version__}")
print(f"✓ IPyWidgets versión: {widgets.__version__}")
print(f"\n✓ Todas las dependencias verificadas. Listo para comenzar.")

## Objetivos de Aprendizaje

Al finalizar esta clase, el estudiante será capaz de:

1. **Comprender** el Tipo de Dato Abstracto (TDA) Fila y su principio FIFO (First In, First Out)
2. **Analizar** las diferencias entre Filas y Pilas en términos de estructura y operaciones
3. **Implementar** una fila utilizando dos enfoques: deque (O(1) en ambos extremos) y lista enlazada (dos punteros)
4. **Aplicar** filas para resolver problemas clásicos: BFS, simulaciones de colas, sistemas de tickets
5. **Evaluar** la complejidad temporal y espacial de operaciones en filas y elegir la implementación adecuada

## Motivación: ¿Por qué necesitamos Filas?

### Analogía: Cola del Banco

Imagina una sucursal bancaria:
- **Cliente 1** llega a las 9:00 AM → entra a la cola
- **Cliente 2** llega a las 9:05 AM → se une a la cola (detrás de Cliente 1)
- **Cliente 3** llega a las 9:10 AM → se une a la cola (detrás de Cliente 2)
- **Cajero abre a las 9:15** → atiende a **Cliente 1** (primero en llegar)
- **Cliente 1 se va** → ahora atienden a **Cliente 2**
- **Cliente 2 se va** → ahora atienden a **Cliente 3**

Este es el principio **FIFO** (First In, First Out): quien llega primero, se atiende primero.

### ¿Cuál es el problema con usar una lista normal?

En Python, si intentamos usar una lista como fila:
```python
cola = [1, 2, 3, 4, 5]
cliente = cola.pop(0)  # ← ¡PROBLEMA: O(n)!
```

El problema es que **`pop(0)` es O(n)**. ¿Por qué? Porque debemos desplazar todos los elementos.

Veamos esto en la práctica:

In [ ]:
# Benchmark: ¿Cuánto tarda pop(0) vs pop() en una lista?

sizes = [100, 500, 1000, 5000, 10000]
times_pop0 = []
times_pop_end = []

print("Comparación de rendimiento: pop(0) vs pop()")
print("=" * 50)

for size in sizes: 
    # Tiempo de pop(0) en lista de tamaño 'size'
    time_pop0 = timeit( 
        lambda: (lista := list(range(size)), lista.pop(0)), ## Usamos walrus operator para crear la lista y luego hacer pop(0)
        number=100
    )
    
    # Tiempo de pop() en lista de tamaño 'size'
    time_pop_end = timeit(
        lambda: (lista := list(range(size)), lista.pop()), ## Usamos walrus operator para crear la lista y luego hacer pop() al final
        number=100
    )
    
    times_pop0.append(time_pop0) 
    times_pop_end.append(time_pop_end)
    
    ratio = time_pop0 / time_pop_end if time_pop_end > 0 else float('inf')
    print(f"Tamaño: {size:5d} | pop(0): {time_pop0:.4f}s | pop(): {time_pop_end:.4f}s | Ratio: {ratio:.1f}x más lento")

print("\n" + "="*50)
print("CONCLUSIÓN: pop(0) es O(n) - se vuelve más lento conforme crece la lista")
print("POR ESO usamos deque (O(1)) o listas enlazadas (O(1) con dos punteros)")

### Pausa para reflexión

**Pregunta para el profesor:** ¿Ven cómo el rendimiento degradó dramáticamente? Esto es crítico en sistemas reales:
- Un servidor de soporte con 10,000 tickets en cola
- Un videojuego con miles de entidades en la cola de renderizado
- Una red de routers con millones de paquetes en cola

En todos estos casos, necesitamos **operaciones O(1)** en ambos extremos. Eso es exactamente lo que nos dan `deque` y las listas enlazadas con dos punteros.

---

# Sección 1: TDA Fila (15 minutos)

## ¿Qué es una Fila?

Una **fila** (queue) es una colección de elementos con las siguientes características:

- **Principio FIFO**: First In, First Out (primero en entrar, primero en salir)
- **Dos extremos**: frente (front) donde se extrae, final (rear/back) donde se inserta
- **Operaciones principales**:
  - `enqueue(item)`: insertar elemento en el final
  - `dequeue()`: extraer elemento del frente
  - `front()`: ver el elemento del frente sin extraer
  - `is_empty()`: verificar si está vacía
  - `size()`: obtener la cantidad de elementos

## Diagrama ASCII: Estructura FIFO

```
FRENTE (extraemos aquí)                      FINAL (insertamos aquí)
   ↓                                               ↓
┌─────┬─────┬─────┬─────┐
│ 10  │ 20  │ 30  │ 40  │
└─────┴─────┴─────┴─────┘
 [0]   [1]   [2]   [3]

Secuencia de operaciones:
1. enqueue(10) → [10]
2. enqueue(20) → [10, 20]
3. enqueue(30) → [10, 20, 30]
4. enqueue(40) → [10, 20, 30, 40]
5. dequeue()   → retorna 10, cola = [20, 30, 40]
6. dequeue()   → retorna 20, cola = [30, 40]
```

## Comparación: Fila vs Pila

| Aspecto | Fila (FIFO) | Pila (LIFO) |
|---------|-------------|-------------|
| **Orden** | Primero en entrar, primero en salir | Último en entrar, primero en salir |
| **Frente** | Un extremo (inicio) | Un extremo (tope) |
| **Final** | Otro extremo (final) | Mismo extremo (tope) |
| **Insertar** | Al final (rear) | Al tope (push) |
| **Extraer** | Del frente (front) | Del tope (pop) |
| **Casos de uso** | BFS, tickets, buffering | DFS, evaluación de expresiones, backtracking |

## Operaciones de la Fila - Tabla Detallada

| Operación | Descripción | Retorna | Complejidad |
|-----------|-------------|---------|-------------|
| `enqueue(item)` | Añade elemento al final | — | O(1) |
| `dequeue()` | Extrae y retorna frente | item o excepción | O(1) |
| `front()` | Retorna frente sin extraer | item o None | O(1) |
| `is_empty()` | ¿Está vacía? | bool | O(1) |
| `size()` | Cantidad de elementos | int | O(1) |


In [ ]:
# Demo básica: FilaDeque

print("DEMO: Creando una fila y realizando operaciones básicas")
print("="*60)

fila = FilaDeque()

print("\n1. Fila vacía:")
print(f"   is_empty() = {fila.is_empty()}") #true si la fila está vacía, false si tiene elementos
print(f"   size() = {fila.size()}") # Numero de elementos en la fila

print("\n2. Enqueue de 4 tickets de soporte:")
tickets = ["TICKET-001", "TICKET-002", "TICKET-003", "TICKET-004"]
for ticket in tickets:
    fila.enqueue(ticket)
    print(f"   enqueue({ticket!r}) → size = {fila.size()}")

print(f"\n3. Estado de la fila:")
print(f"   size() = {fila.size()}")
print(f"   front() = {fila.front()!r}") ## !r invoca a repr() para mostrar la representación de cadena del elemento, útil para strings. repr() es una función que devuelve una representación de cadena de un objeto, a menudo más detallada o precisa que str(). En este caso, muestra el ticket con comillas, indicando que es una cadena.

print(f"\n4. Dequeue - atendiendo clientes en orden FIFO:")
while not fila.is_empty():
    cliente = fila.dequeue()
    print(f"   dequeue() → Atendiendo: {cliente!r}, quedan {fila.size()} en cola")

print(f"\n5. Fila vacía nuevamente:")
print(f"   is_empty() = {fila.is_empty()}")

# Sección 2: Implementación con Deque (20 minutos)

## ¿Qué es `collections.deque`?

`deque` (Double-Ended Queue) es una estructura de datos especial en Python que permite:
- **Inserción y extracción O(1)** en AMBOS extremos
- Mantiene referencias internas optimizadas (internamente es una lista doblemente enlazada)
- Mucho más eficiente que usar `list.pop(0)`

## Diagrama de Memoria: FilaDeque

```
Internamente, deque mantiene bloques de memoria:

┌──────────────────────────────────────────┐
│           deque([10, 20, 30, 40])         │
├──────────────────────────────────────────┤
│  FRENTE                              FINAL│
│    ↓                                    ↓ │
│  ┌──┐ → ┌──┐ → ┌──┐ → ┌──┐            │
│  │10│   │20│   │30│   │40│  ←→ block  │
│  └──┘ ← └──┘ ← └──┘ ← └──┘            │
└──────────────────────────────────────────┘

Operaciones:
- append(x):        O(1) amortizado - inserta al final
- appendleft(x):    O(1) amortizado - inserta al inicio
- pop():            O(1) amortizado - extrae del final
- popleft():        O(1) amortizado - extrae del inicio (ESTO es FRENTE)
```

## Código de FilaDeque

Supongamos que `filas.py` contiene:

In [ ]:
# Mostremos el código esperado de FilaDeque
# (en un archivo real sería en filas.py)

codigo_FilaDeque = '''
from collections import deque
from abc import ABC, abstractmethod

class FilaTDA(ABC):
    """Tipo de Dato Abstracto para Fila (Queue - FIFO)"""
    
    @abstractmethod
    def enqueue(self, item):
        """Insertar elemento al final de la fila"""
        pass
    
    @abstractmethod
    def dequeue(self):
        """Extraer y retornar elemento del frente"""
        pass
    
    @abstractmethod
    def front(self):
        """Retornar elemento del frente sin extraer"""
        pass
    
    @abstractmethod
    def is_empty(self):
        """Verificar si la fila está vacía"""
        pass
    
    @abstractmethod
    def size(self):
        """Retornar cantidad de elementos"""
        pass


class FilaDeque(FilaTDA):
    """Implementación de Fila usando collections.deque (O(1) en ambos extremos)"""
    
    def __init__(self):
        self.__elementos = deque() ## deque esta implementada en la biblioteca estándar de Python y es una estructura de datos optimizada para operaciones en ambos extremos, lo que la hace ideal para implementar una fila (queue) con eficiencia O(1) tanto para enqueue como para dequeue. se utiliza el principio de encapsulamiento al usar un atributo privado __elementos para almacenar los datos de la fila, lo que ayuda a proteger la integridad de la estructura y evita modificaciones externas no controladas.
    
    def enqueue(self, item):
        """Insertar elemento al final: O(1)"""
        self.__elementos.append(item)
    
    def dequeue(self):
        """Extraer del frente: O(1)"""
        if self.is_empty():
            raise IndexError("dequeue de fila vacía")
        return self.__elementos.popleft()
    
    def front(self):
        """Ver frente sin extraer: O(1)"""
        if self.is_empty():
            return None
        return self.__elementos[0]
    
    def is_empty(self):
        """Verificar si está vacía: O(1)"""
        return len(self.__elementos) == 0
    
    def size(self):
        """Obtener tamaño: O(1)"""
        return len(self.__elementos)
'''

print("Estructura de FilaDeque (collections.deque como backend):")
print("="*70)
print(codigo_FilaDeque)
print("="*70)

In [ ]:
# Demo paso a paso: FilaDeque

print("DEMO: FilaDeque paso a paso")
print("="*70)

fila = FilaDeque() # fila doblemente encapsulada, con operaciones O(1) garantizadas por deque

operaciones = [
    ("enqueue(100)", lambda: fila.enqueue(100)),
    ("enqueue(200)", lambda: fila.enqueue(200)),
    ("enqueue(300)", lambda: fila.enqueue(300)),
    ("front()", lambda: fila.front()),
    ("dequeue()", lambda: fila.dequeue()),
    ("size()", lambda: fila.size()),
    ("front()", lambda: fila.front()),
    ("dequeue()", lambda: fila.dequeue()),
    ("dequeue()", lambda: fila.dequeue()),
    ("is_empty()", lambda: fila.is_empty()),
]

for paso, (operacion, func) in enumerate(operaciones, 1):
    resultado = func()
    estado_actual = f"size={fila.size()}, front={fila.front()}"
    print(f"{paso:2d}. {operacion:20s} → {str(resultado):10s} | Estado: {estado_actual}")

In [ ]:
# Visualización con matplotlib: Secuencia de operaciones FilaDeque

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('FilaDeque: Secuencia de operaciones FIFO', fontsize=14, fontweight='bold')

def dibujar_fila(ax, elementos, titulo, operacion=""):
    """Función auxiliar para dibujar una fila"""
    ax.set_xlim(-0.5, 5)
    ax.set_ylim(-0.5, 2)
    ax.axis('off')
    ax.set_title(titulo, fontweight='bold', fontsize=11)
    
    # Etiquetas
    if elementos:
        ax.text(-0.4, 1.5, 'FRENTE', fontsize=9, fontweight='bold', color='green')
        ax.text(3.8, 1.5, 'FINAL', fontsize=9, fontweight='bold', color='orange')
    
    # Dibujar elementos
    for i, elem in enumerate(elementos):
        color = 'lightgreen' if i == 0 else 'lightblue' if i < len(elementos)-1 else 'lightyellow'
        rect = FancyBboxPatch((i*0.9, 0.5), 0.8, 0.6, 
                              boxstyle="round,pad=0.05", 
                              edgecolor='black', facecolor=color, linewidth=2)
        ax.add_patch(rect)
        ax.text(i*0.9+0.4, 0.8, str(elem), ha='center', va='center', 
               fontsize=11, fontweight='bold')
        
        if i < len(elementos)-1:
            arrow = FancyArrowPatch((i*0.9+0.8, 0.8), ((i+1)*0.9, 0.8),
                                   arrowstyle='->', mutation_scale=15, color='black')
            ax.add_patch(arrow)
    
    # Operación
    if operacion:
        ax.text(2.5, 0.05, f"Operación: {operacion}", fontsize=9, 
               ha='center', style='italic', color='darkred')
    
    if not elementos:
        ax.text(2.5, 0.8, '[vacía]', ha='center', va='center', 
               fontsize=12, style='italic', color='gray')

estados = [
    ([], "1. Inicio: Fila vacía", ""),
    ([100], "2. enqueue(100)", "enqueue(100)"),
    ([100, 200, 300], "3. enqueue(200, 300)", "enqueue(200), enqueue(300)"),
    ([100, 200, 300], "4. front() = 100", "front()"),
    ([200, 300], "5. dequeue() = 100", "dequeue()"),
    ([300], "6. dequeue() = 200", "dequeue()"),
]

for idx, (ax, (elementos, titulo, operacion)) in enumerate(zip(axes.flat, estados)):
    dibujar_fila(ax, elementos, titulo, operacion)

plt.tight_layout()
plt.savefig('/tmp/fila_secuencia_deque.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nVisualización: FilaDeque con secuencia FIFO")

# Sección 3: Implementación con Lista Enlazada (20 minutos)

## ¿Por qué una lista enlazada?

Aunque `deque` es excelente, las listas enlazadas permiten:
- **Comprender la estructura interna** de una fila
- **Aprender sobre punteros** (conceptual importante)
- **Implementar en otros lenguajes** donde no exista `deque` nativo

## Diferencia crítica: DOS punteros vs UNO

### PilaEnlazada (que ya conocen):
```
         __tope
           ↓
   ┌───┐  ┌───┐  ┌───┐
   │ 1 │→ │ 2 │→ │ 3 │→ None
   └───┘  └───┘  └───┘

- Un solo puntero en el TOPE
- push() y pop() en el TOPE → O(1)
- No necesitamos acceso al final (rear)
```

### FilaEnlazada (nueva):
```
    __frente                           __final
       ↓                                 ↓
   ┌───┐  ┌───┐  ┌───┐  ┌───┐
   │ 1 │→ │ 2 │→ │ 3 │→ │ 4 │→ None
   └───┘  └───┘  └───┘  └───┘

- DOS punteros: frente (inicio) y final (rear)
- enqueue() en el FINAL → O(1)
- dequeue() en el FRENTE → O(1)
- Ambas operaciones críticas son O(1)
```

## Código de FilaEnlazada

Supongamos que `filas.py` también contiene:

In [ ]:
codigo_FilaEnlazada = '''
class _Nodo:
    """Nodo para lista enlazada"""
    def __init__(self, dato):
        self.dato = dato
        self.siguiente = None


class FilaEnlazada(FilaTDA):
    """Implementación de Fila usando Lista Enlazada (DOS punteros)"""
    
    def __init__(self):
        self.__frente = None   # Puntero al inicio
        self.__final = None    # Puntero al final
        self.__cantidad = 0
    
    def enqueue(self, item):
        """Insertar al final: O(1)"""
        nuevo_nodo = _Nodo(item)
        
        if self.is_empty():
            self.__frente = nuevo_nodo
        else:
            self.__final.siguiente = nuevo_nodo ## Si la fila no está vacía, el nodo actual al final apunta al nuevo nodo
        
        self.__final = nuevo_nodo # El nuevo nodo ahora es el final de la fila
        self.__cantidad += 1 
    
    def dequeue(self):
        """Extraer del frente: O(1)"""
        if self.is_empty():
            raise IndexError("dequeue de fila vacía")
        
        dato = self.__frente.dato
        self.__frente = self.__frente.siguiente
        self.__cantidad -= 1
        
        if self.is_empty(): 
            self.__final = None ## Esta linea solo tiene sentido cuando queda un solo elemento, porque al hacer dequeue() el frente se vuelve None, entonces el final también debe ser None para mantener la consistencia de la estructura.
        
        return dato
    
    def front(self):
        """Ver frente sin extraer: O(1)"""
        if self.is_empty():
            return None
        return self.__frente.dato
    
    def is_empty(self):
        """Verificar si está vacía: O(1)"""
        return self.__frente is None
    
    def size(self):
        """Obtener tamaño: O(1)"""
        return self.__cantidad
'''

print("Estructura de FilaEnlazada (DOS punteros: __frente y __final):")
print("="*70)
print(codigo_FilaEnlazada)
print("="*70)

In [ ]:
# Demo con FilaEnlazada

print("DEMO: FilaEnlazada (idéntica API que FilaDeque)")
print("="*70)

fila_enlazada = FilaEnlazada()

print("\nOperaciones iniciales:")
operaciones = [
    ("enqueue(100)", lambda: fila_enlazada.enqueue(100)),
    ("enqueue(200)", lambda: fila_enlazada.enqueue(200)),
    ("enqueue(300)", lambda: fila_enlazada.enqueue(300)),
    ("size()", lambda: fila_enlazada.size()),
    ("front()", lambda: fila_enlazada.front()),
]

for operacion, func in operaciones:
    resultado = func()
    print(f"  {operacion:20s} → {resultado}")

print("\nExtrayendo (FIFO):")
while not fila_enlazada.is_empty():
    valor = fila_enlazada.dequeue()
    print(f"  dequeue() → {valor}, quedan {fila_enlazada.size()} elementos")

print("\nFila vacía:", fila_enlazada.is_empty())

In [ ]:
# Visualización: Diagrama de memoria de FilaEnlazada

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Diagrama 1: Fila vacía
ax = axes[0]
ax.set_xlim(-0.5, 5)
ax.set_ylim(-0.5, 3)
ax.axis('off')
ax.set_title('FilaEnlazada vacía', fontsize=12, fontweight='bold')

# Punteros
ax.text(0.5, 2.5, '__frente = None', fontsize=10, fontweight='bold', color='darkred')
ax.text(0.5, 2.0, '__final = None', fontsize=10, fontweight='bold', color='darkorange')
ax.text(2, 1.2, '[vacía]', fontsize=14, style='italic', color='gray', ha='center')

# Diagrama 2: Fila con elementos
ax = axes[1]
ax.set_xlim(-0.5, 6.5)
ax.set_ylim(-0.5, 3)
ax.axis('off')
ax.set_title('FilaEnlazada después de enqueue(10,20,30)', fontsize=12, fontweight='bold')

# Punteros
ax.annotate('__frente', xy=(0.3, 2.2), xytext=(-0.3, 2.5),
           arrowprops=dict(arrowstyle='->', color='darkred', lw=2),
           fontsize=10, fontweight='bold', color='darkred')
ax.annotate('__final', xy=(3.8, 2.2), xytext=(4.2, 2.5),
           arrowprops=dict(arrowstyle='->', color='darkorange', lw=2),
           fontsize=10, fontweight='bold', color='darkorange')

# Nodos
nodos = [(0.3, '10'), (1.5, '20'), (2.7, '30')]
for i, (x, valor) in enumerate(nodos):
    # Caja del nodo
    rect = FancyBboxPatch((x-0.25, 1.8), 0.5, 0.7,
                          boxstyle="round,pad=0.05",
                          edgecolor='black', facecolor='lightblue', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, 2.15, valor, ha='center', va='center', fontsize=11, fontweight='bold')
    
    # Flecha al siguiente
    if i < len(nodos) - 1:
        arrow = FancyArrowPatch((x+0.25, 2.15), (nodos[i+1][0]-0.25, 2.15),
                               arrowstyle='->', mutation_scale=20, color='black', lw=1.5)
        ax.add_patch(arrow)
        ax.text(x+0.55, 2.35, 'sig', fontsize=8, color='gray')
    else:
        ax.text(x+0.4, 2.15, 'None', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig('/tmp/fila_enlazada_memoria.png', dpi=100, bbox_inches='tight')
plt.show()

print("Visualización: Estructura de memoria de FilaEnlazada con DOS punteros")

# Sección 4: Aplicaciones Clásicas (25 minutos)

Las filas son fundamentales en computación. Veamos tres aplicaciones clásicas:

## Aplicación 1: Simulación de Cola de Impresión

En una oficina con una impresora compartida:
1. Usuario A envía documento de 5 páginas
2. Usuario B envía documento de 3 páginas
3. Usuario C envía documento de 8 páginas
4. Impresora imprime en orden FIFO

La fila maneja la **equidad**: quien llega primero, imprime primero.

In [ ]:
# Aplicación 1: Cola de Impresión

class TrabajoDImpresion:
    def __init__(self, id_usuario, num_paginas):
        self.id_usuario = id_usuario
        self.num_paginas = num_paginas
        self.timestamp = time.time()
    
    def __repr__(self): ## Representación de cadena para mostrar el trabajo de impresión
        return f"Trabajo(usuario={self.id_usuario}, páginas={self.num_paginas})"

def simular_cola_impresion():
    """Simula una cola de impresión FIFO"""
    print("SIMULACIÓN: Cola de Impresión")
    print("="*70)
    
    cola_impresion = FilaDeque() ## Usamos FilaDeque para garantizar O(1) en enqueue y dequeue, lo que es crucial para una cola de impresión eficiente.
    tiempo_simulacion = 0
    
    # Trabajos llegan en orden
    trabajos_llegada = [
        ("Usuario A", 5),
        ("Usuario B", 3),
        ("Usuario C", 8),
        ("Usuario D", 2),
    ]
    
    print("\n1. LLEGADA DE TRABAJOS:")
    for usuario, paginas in trabajos_llegada: # Simulamos la llegada de trabajos en orden, cada trabajo tiene un número de páginas que determina su tiempo de impresión.
        trabajo = TrabajoDImpresion(usuario, paginas)
        cola_impresion.enqueue(trabajo)
        print(f"   {tiempo_simulacion}s - {usuario} encolado: {paginas} páginas (cola: {cola_impresion.size()})")
        tiempo_simulacion += 1
    
    print(f"\n2. IMPRESIÓN (en orden FIFO):")
    tiempo_impresion = 0.5  # segundos por página
    numero_orden = 1
    
    while not cola_impresion.is_empty():
        trabajo = cola_impresion.dequeue()
        tiempo_impresion_trabajo = trabajo.num_paginas * tiempo_impresion # tiempo de impresión basado en número de páginas
        tiempo_simulacion += tiempo_impresion_trabajo
        
        print(f"   [{numero_orden}] Imprimiendo {trabajo}: {tiempo_impresion_trabajo:.1f}s (tiempo total: {tiempo_simulacion:.1f}s)")
        numero_orden += 1
    
    print(f"\n3. COLA VACÍA")
    print(f"   Tiempo total de simulación: {tiempo_simulacion:.1f} segundos")
    print(f"   ✓ Todos los trabajos completados en orden FIFO")

simular_cola_impresion()

## Aplicación 2: BFS (Breadth-First Search) en Grilla 2D

**Problema:** Encontrar el camino más corto en una grilla de un punto A a un punto B.

**Por qué usamos fila:** BFS explora "por capas" (todos los vecinos a distancia 1, luego todos a distancia 2, etc.). Una fila FIFO es perfecta para esto.

**Algoritmo:**
1. Enqueue(inicio)
2. Mientras fila no esté vacía:
   - Dequeue nodo actual
   - Si es el destino, fin
   - Enqueue todos los vecinos no visitados

In [ ]:
# Aplicación 2: BFS en Grilla 2D

def bfs_grilla(grilla, inicio, destino):
    """
    Busca camino más corto en grilla usando BFS.
    grilla: matriz donde 0=transitable, 1=obstáculo
    inicio, destino: tuplas (fila, col)
    Retorna: lista de pasos si existe camino, None si no
    """
    filas, cols = len(grilla), len(grilla[0])
    visitado = set()
    padre = {}  # para reconstruir camino
    cola = FilaDeque()
    
    cola.enqueue(inicio)
    visitado.add(inicio)
    padre[inicio] = None
    
    # Direcciones: arriba, derecha, abajo, izquierda
    direcciones = [(-1, 0), (0, 1), (1, 0), (0, -1)]
    
    while not cola.is_empty():
        fila, col = cola.dequeue()
        
        if (fila, col) == destino:
            # Reconstruir camino
            camino = []
            pos = destino
            while pos is not None:
                camino.append(pos)
                pos = padre[pos]
            return camino[::-1]
        
        # Explorar vecinos
        for df, dc in direcciones:
            nf, nc = fila + df, col + dc
            if (0 <= nf < filas and 0 <= nc < cols and 
                grilla[nf][nc] == 0 and (nf, nc) not in visitado):
                visitado.add((nf, nc)) ## Marcamos el nodo como visitado
                padre[(nf, nc)] = (fila, col) # Guardamos el padre para reconstruir el camino luego
                cola.enqueue((nf, nc))
    
    return None  # No hay camino


print("APLICACIÓN 2: BFS (Breadth-First Search) en Grilla")
print("="*70)

# Crear grilla de prueba: 0=transitable, 1=obstáculo
grilla = [
    [0, 0, 0, 1, 0],
    [1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 1],
    [0, 0, 0, 0, 0],
]

inicio = (0, 0)
destino = (4, 4)

print(f"\nGrilla 5x5 (0=transitable, 1=obstáculo):")
for i, fila_grilla in enumerate(grilla):
    print(f"  {fila_grilla}")

print(f"\nBuscando camino de {inicio} a {destino} usando BFS con fila...\n")

camino = bfs_grilla(grilla, inicio, destino)

if camino:
    print(f"✓ Camino encontrado (longitud: {len(camino)} pasos):")
    for i, paso in enumerate(camino):
        print(f"  Paso {i}: {paso}")
else:
    print("✗ No hay camino disponible")

In [ ]:
# Visualizar BFS en grilla con matplotlib

fig, ax = plt.subplots(figsize=(8, 8))

grilla_visual = [
    [0, 0, 0, 1, 0],
    [1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 1],
    [0, 0, 0, 0, 0],
]

inicio = (0, 0)
destino = (4, 4)
camino_resultado = [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]

filas, cols = len(grilla_visual), len(grilla_visual[0])

for i in range(filas):
    for j in range(cols):
        if grilla_visual[i][j] == 1:  # Obstáculo
            color = 'black'
        elif (i, j) == inicio:  # Inicio
            color = 'green'
        elif (i, j) == destino:  # Destino
            color = 'red'
        elif (i, j) in camino_resultado:  # Camino
            color = 'yellow'
        else:  # Transitable
            color = 'white'
        
        rect = mpatches.Rectangle((j, filas-1-i), 1, 1, 
                                 linewidth=2, edgecolor='gray', 
                                 facecolor=color)
        ax.add_patch(rect)
        ax.text(j+0.5, filas-1-i+0.5, f'({i},{j})', 
               ha='center', va='center', fontsize=8)

ax.set_xlim(0, cols)
ax.set_ylim(0, filas)
ax.set_aspect('equal')
ax.invert_yaxis()
ax.set_title('BFS: Encontrando camino más corto en grilla\nVerde=Inicio, Rojo=Destino, Amarillo=Camino, Negro=Obstáculo', 
            fontweight='bold', fontsize=11)
ax.axis('off')

plt.tight_layout()
plt.savefig('/tmp/bfs_grilla.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\nVisualización: Camino más corto encontrado por BFS = {len(camino_resultado)-1} pasos")

## Aplicación 3: Sistema de Tickets de Soporte

Un sistema típico de soporte técnico:
- Tickets llegan continuamente
- Se atienden en orden FIFO (equitativo)
- Se registra timestamp de llegada y tiempo de resolución

In [ ]:
# Aplicación 3: Sistema de Tickets de Soporte

import random
from datetime import datetime, timedelta

class TicketSoporte:
    contador = 1
    
    def __init__(self, cliente, asunto, prioridad='normal'):
        self.id = TicketSoporte.contador
        TicketSoporte.contador += 1
        self.cliente = cliente
        self.asunto = asunto
        self.prioridad = prioridad
        self.timestamp_llegada = datetime.now()
        self.timestamp_resolucion = None
    
    def tiempo_espera(self):
        if self.timestamp_resolucion:
            delta = self.timestamp_resolucion - self.timestamp_llegada
            return delta.total_seconds()
        return None
    
    def __repr__(self):
        return f"TICKET-{self.id:04d} ({self.cliente}): {self.asunto}"

def simular_soporte():
    print("APLICACIÓN 3: Sistema de Tickets de Soporte")
    print("="*70)
    
    cola_tickets = FilaDeque()
    tickets_atendidos = []
    
    clientes = ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve']
    asuntos = ['Login no funciona', 'Error de pago', 'Resetear contraseña', 
              'Acceso denegado', 'Datos no sincronizar']
    
    print("\n1. LLEGADA DE TICKETS (simulada):")
    for i in range(6):
        cliente = random.choice(clientes)
        asunto = random.choice(asuntos)
        ticket = TicketSoporte(cliente, asunto)
        cola_tickets.enqueue(ticket)
        print(f"   {i+1}. {ticket} - Cola: {cola_tickets.size()} tickets")
    
    print(f"\n2. ATENCIÓN DE TICKETS (FIFO):")
    tiempo_actual = datetime.now()
    numero_atencion = 1
    
    while not cola_tickets.is_empty():
        ticket = cola_tickets.dequeue()
        # Simular tiempo de resolución (1-3 minutos)
        tiempo_resolucion = random.uniform(1, 3)
        ticket.timestamp_resolucion = tiempo_actual + timedelta(minutes=tiempo_resolucion)
        tickets_atendidos.append(ticket)
        
        espera = ticket.tiempo_espera() / 60  # en minutos
        print(f"   [{numero_atencion}] {ticket}")
        print(f"       Tiempo de espera: {espera:.1f} min, Resolución: {tiempo_resolucion:.1f} min")
        numero_atencion += 1
    
    # Estadísticas
    print(f"\n3. ESTADÍSTICAS:")
    tiempo_espera_promedio = sum(t.tiempo_espera() for t in tickets_atendidos) / len(tickets_atendidos) / 60
    print(f"   Total de tickets atendidos: {len(tickets_atendidos)}")
    print(f"   Tiempo promedio de espera: {tiempo_espera_promedio:.1f} minutos")
    print(f"   ✓ Sistema FIFO garantiza equidad")

simular_soporte()

# Sección 5: Análisis de Complejidad (10 minutos)

In [ ]:
!pip install pandas

In [ ]:
# Análisis de Complejidad: Tabla Comparativa

import pandas as pd

data = {
    'Operación': ['enqueue(item)', 'dequeue()', 'front()', 'is_empty()', 'size()'],
    'FilaDeque': ['O(1)*', 'O(1)*', 'O(1)', 'O(1)', 'O(1)'],
    'FilaEnlazada': ['O(1)', 'O(1)', 'O(1)', 'O(1)', 'O(1)'],
    'list.append() + list.pop(0)': ['O(1)', 'O(n)❌', 'O(1)', 'O(1)', 'O(1)'],
}

df = pd.DataFrame(data)
print("\nTABLA DE COMPLEJIDAD TEMPORAL:")
print("="*80)
print(df.to_string(index=False))
print("="*80)
print("\nNotas:")
print("  * O(1) amortizado (las operaciones al final de deque son realmente rápidas)")
print("  ❌ O(n) es INACEPTABLE para colas grandes - todos los elementos se desplazan")
print("\nComplejidad Espacial:")
print("  - FilaDeque: O(n) para n elementos")
print("  - FilaEnlazada: O(n) para n elementos (con overhead de punteros)")

In [ ]:
# Benchmark: Comparación de rendimiento FilaDeque vs FilaEnlazada vs list

print("\nBENCHMARK: Rendimiento en operaciones enqueue + dequeue")
print("="*70)

sizes = [100, 500, 1000, 5000]
resultados = {'size': [], 'deque': [], 'enlazada': [], 'list.pop(0)': []}

for size in sizes:
    # FilaDeque
    time_deque = timeit(
        lambda: (f := FilaDeque(), [f.enqueue(i) for i in range(size)], 
                [f.dequeue() for _ in range(size)]),
        number=100
    )
    
    # FilaEnlazada
    time_enlazada = timeit(
        lambda: (f := FilaEnlazada(), [f.enqueue(i) for i in range(size)], 
                [f.dequeue() for _ in range(size)]),
        number=100
    )
    
    # list con pop(0)
    time_list = timeit(
        lambda: (l := [], [l.append(i) for i in range(size)], 
                [l.pop(0) for _ in range(size)]),
        number=100
    )
    
    resultados['size'].append(size)
    resultados['deque'].append(time_deque)
    resultados['enlazada'].append(time_enlazada)
    resultados['list.pop(0)'].append(time_list)
    
    ratio_lista = time_list / time_deque if time_deque > 0 else 0
    print(f"Tamaño {size:5d} | deque: {time_deque:.4f}s | enlazada: {time_enlazada:.4f}s | list: {time_list:.4f}s ({ratio_lista:.1f}x lento)")

print("\nCONCLUSIÓN: Usar deque o lista enlazada, NUNCA list.pop(0) para colas")

In [ ]:
# Gráfico de complejidad

fig, ax = plt.subplots(figsize=(10, 6))

colores = {'deque': 'green', 'enlazada': 'blue', 'list.pop(0)': 'red'}

for metodo, color in colores.items():
    ax.plot(resultados['size'], resultados[metodo], 
           marker='o', linewidth=2.5, label=metodo, color=color, markersize=8)

ax.set_xlabel('Tamaño de Fila (n)', fontsize=11, fontweight='bold')
ax.set_ylabel('Tiempo (segundos)', fontsize=11, fontweight='bold')
ax.set_title('Benchmark: Rendimiento de Implementaciones de Fila\n(100 iteraciones de enqueue + dequeue)', 
            fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)

# Anotación
ax.text(3500, resultados['list.pop(0)'][-1] * 0.8, 
       '❌ list.pop(0) es O(n)\ny se vuelve exponencialmente lento',
       fontsize=10, color='red', fontweight='bold',
       bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

plt.tight_layout()
plt.savefig('/tmp/benchmark_colas.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nGráfico: Comparación de rendimiento temporal")

# Sección 6: Widget Interactivo - Simulador Visual de Fila

In [ ]:
# Widget interactivo para simular una fila

class SimuladorFila:
    def __init__(self):
        self.fila = FilaDeque()
        self.historial = []
        
        # Widgets
        self.input_valor = widgets.IntText(value=0, description='Valor:', width='100px')
        self.btn_enqueue = widgets.Button(description='Enqueue', button_style='info')
        self.btn_dequeue = widgets.Button(description='Dequeue', button_style='warning')
        self.btn_limpiar = widgets.Button(description='Limpiar', button_style='danger')
        self.output = widgets.Output()
        
        # Event handlers
        self.btn_enqueue.on_click(self._enqueue)
        self.btn_dequeue.on_click(self._dequeue)
        self.btn_limpiar.on_click(self._limpiar)
    
    def _enqueue(self, b):
        valor = self.input_valor.value
        self.fila.enqueue(valor)
        self.historial.append(f"enqueue({valor})")
        self._actualizar_display()
    
    def _dequeue(self, b):
        try:
            valor = self.fila.dequeue()
            self.historial.append(f"dequeue() → {valor}")
        except IndexError:
            self.historial.append("dequeue() → ERROR (fila vacía)")
        self._actualizar_display()
    
    def _limpiar(self, b):
        self.fila = FilaDeque()
        self.historial = []
        self._actualizar_display()
    
    def _actualizar_display(self):
        with self.output:
            clear_output(wait=True)
            print("\n📊 SIMULADOR INTERACTIVO DE FILA (FIFO)")
            print("="*60)
            print(f"\n📋 Estado actual:")
            print(f"   Tamaño: {self.fila.size()}")
            print(f"   Frente: {self.fila.front()}")
            print(f"   ¿Vacía?: {self.fila.is_empty()}")
            
            print(f"\n📝 Historial de operaciones:")
            for i, op in enumerate(self.historial[-10:], 1):  # Últimas 10
                print(f"   {i}. {op}")
            
            if self.historial:
                print(f"\n   ... mostrando últimas {min(10, len(self.historial))} operaciones")
    
    def mostrar(self):
        self._actualizar_display()
        display(widgets.VBox([
            widgets.HTML("<h3>🎮 Simulador de Fila - Enqueue / Dequeue</h3>"),
            widgets.HBox([self.input_valor, self.btn_enqueue, self.btn_dequeue, self.btn_limpiar]),
            self.output
        ]))

# Crear y mostrar simulador
simulador = SimuladorFila()
simulador.mostrar()

print("\n✓ Widget interactivo listo. Prueba enqueue, dequeue y limpiar.")

# Sección 7: Ejercicios Prácticos

## Ejercicio 1 ⭐: Invertir una Fila usando una Pila

**Problema:** Dada una fila [1, 2, 3, 4], produce [4, 3, 2, 1] utilizando una pila auxiliar.

**Pista:** Las pilas invierten el orden (LIFO). ¿Cómo combinar dos inversiones para lograr el objetivo?

In [ ]:
# Ejercicio 1: Invertir fila

def invertir_fila(fila):
    """
    Invierte una fila [1,2,3,4] → [4,3,2,1]
    Usando una pila auxiliar.
    
    Proceso:
    1. Vaciar fila en pila (invierte una vez: [4,3,2,1])
    2. Vaciar pila en fila (invierte otra vez: [1,2,3,4] pero al revés)
    3. Vaciar fila en pila (invierte una vez más: [4,3,2,1])
    """
    from pilas import PilaDeque  # Asumiendo que existe
    
    pila = []  # Usamos una lista como pila para simplicidad
    
    # Paso 1: Vaciar fila en pila
    while not fila.is_empty():
        pila.append(fila.dequeue())
    
    # Paso 2: Vaciar pila en fila
    while pila:
        fila.enqueue(pila.pop())
    
    # Paso 3: Vaciar fila en pila
    while not fila.is_empty():
        pila.append(fila.dequeue())
    
    # Paso 4: Vaciar pila en fila (ahora invertida)
    while pila:
        fila.enqueue(pila.pop())
    
    return fila

print("EJERCICIO 1 ⭐: Invertir una Fila")
print("="*70)

fila_original = FilaDeque()
for val in [1, 2, 3, 4]:
    fila_original.enqueue(val)

print(f"\nFila original:")
fila_temp = FilaDeque()
vals_orig = []
while not fila_original.is_empty():
    v = fila_original.dequeue()
    vals_orig.append(v)
    fila_temp.enqueue(v)
fila_original = fila_temp
print(f"  {vals_orig}")

fila_invertida = invertir_fila(fila_original)

vals_inv = []
while not fila_invertida.is_empty():
    vals_inv.append(fila_invertida.dequeue())

print(f"\nFila invertida:")
print(f"  {vals_inv}")
print(f"\n✓ Solución correcta: {vals_inv == [4, 3, 2, 1]}")

## Ejercicio 2 ⭐⭐: Detector de Palíndromo con Fila + Pila

**Problema:** Determinar si una secuencia de números es un palíndromo (se lee igual al revés) usando una fila y una pila.

**Algoritmo:**
1. Insertar todos los elementos en FILA y PILA
2. Extraer simultáneamente: si coinciden, es palíndromo

In [ ]:
def es_palindromo(secuencia):
    """
    Determina si una secuencia es palíndromo.
    Usa fila (FIFO) y pila (LIFO) para comparar.
    """
    fila = FilaDeque()
    pila = []  # Lista como pila
    
    # Llenar ambas estructuras
    for elem in secuencia:
        fila.enqueue(elem)
        pila.append(elem)
    
    # Comparar extrayendo: fila (inicio), pila (final)
    while not fila.is_empty():
        if fila.dequeue() != pila.pop():
            return False
    
    return True

print("EJERCICIO 2 ⭐⭐: Palíndromo con Fila + Pila")
print("="*70)

test_cases = [
    [1, 2, 3, 2, 1],      # Sí es palíndromo
    [5, 4, 3, 4, 5],     # Sí es palíndromo
    [1, 2, 3, 4, 5],     # No es palíndromo
    ['a', 'b', 'c', 'b', 'a'],  # Sí es palíndromo
]

for secuencia in test_cases:
    resultado = es_palindromo(secuencia)
    print(f"\n  {secuencia}")
    print(f"  → ¿Palíndromo?: {'✓ SÍ' if resultado else '✗ NO'}")

## Ejercicio 3 ⭐⭐⭐: Sliding Window Maximum

**Problema:** Dada una lista y un tamaño de ventana k, encontrar el máximo en cada ventana deslizante.

**Ejemplo:** `[3, 1, 2, 4, 5, 2, 3, 1]` con k=3 → `[3, 4, 5, 5, 5, 3]`

**Estrategia:** Usar una fila de deque para mantener los índices en orden decreciente de valores.

In [ ]:
def sliding_window_maximum(arr, k):
    """
    Encuentra el máximo en cada ventana deslizante de tamaño k.
    Usa deque para O(n) en lugar de O(n*k).
    """
    if not arr or k == 0:
        return []
    
    from collections import deque
    dq = deque()  # Almacena índices
    resultado = []
    
    for i in range(len(arr)):
        # Eliminar índices fuera de ventana
        while dq and dq[0] < i - k + 1:
            dq.popleft()
        
        # Eliminar elementos menores que el actual
        while dq and arr[dq[-1]] < arr[i]:
            dq.pop()
        
        dq.append(i)
        
        # Cuando la ventana está completa, el frente es el máximo
        if i >= k - 1:
            resultado.append(arr[dq[0]])
    
    return resultado

print("EJERCICIO 3 ⭐⭐⭐: Sliding Window Maximum")
print("="*70)

test_cases = [
    ([3, 1, 2, 4, 5, 2, 3, 1], 3),
    ([1, 3, 1, 2, 0, 5], 3),
    ([9, 11], 2),
]

for arr, k in test_cases:
    resultado = sliding_window_maximum(arr, k)
    print(f"\n  Array: {arr}")
    print(f"  Ventana k={k}")
    print(f"  Máximos: {resultado}")

# Sección 8: Autoevaluación (IPyWidgets)

In [ ]:
# Quiz interactivo de autoevaluación

class QuizFifo:
    def __init__(self):
        self.respuestas_usuario = {}
        self.preguntas = [
            {
                'id': 1,
                'pregunta': '¿Cuál es el orden de una fila?',
                'opciones': ['LIFO (Last In First Out)', 'FIFO (First In First Out)', 'Random', 'FILO (First In Last Out)'],
                'correcta': 1,
            },
            {
                'id': 2,
                'pregunta': '¿Cuál es la complejidad de dequeue() en FilaDeque?',
                'opciones': ['O(n)', 'O(log n)', 'O(1)', 'O(n²)'],
                'correcta': 2,
            },
            {
                'id': 3,
                'pregunta': '¿Cuántos punteros tiene FilaEnlazada?',
                'opciones': ['0', '1 (solo frente)', '2 (frente y final)', '3 (frente, final y medio)'],
                'correcta': 2,
            },
            {
                'id': 4,
                'pregunta': '¿Para qué algoritmo es fundamental usar una fila?',
                'opciones': ['DFS (Depth-First Search)', 'BFS (Breadth-First Search)', 'Búsqueda binaria', 'Ordenamiento rápido'],
                'correcta': 1,
            },
        ]
    
    def crear_quiz(self):
        output = widgets.Output()
        botones_respuesta = {}
        
        def actualizar_resultado(b):
            with output:
                clear_output(wait=True)
                aciertos = sum(1 for i, preg in enumerate(self.preguntas)
                              if i in botones_respuesta and botones_respuesta[i].value == preg['correcta'])
                total = len(self.preguntas)
                porcentaje = (aciertos / total) * 100
                
                print(f"\n📊 RESULTADOS DE AUTOEVALUACIÓN")
                print("="*50)
                print(f"\nPuntuación: {aciertos}/{total} ({porcentaje:.0f}%)")
                
                if porcentaje >= 80:
                    print("\n🎉 ¡Excelente! Dominas bien el tema de Filas")
                elif porcentaje >= 60:
                    print("\n✓ Buen desempeño. Repasa los conceptos de complejidad")
                else:
                    print("\n⚠ Necesitas repasar. Enfócate en FIFO vs LIFO y complejidad")
                
                print("\nDetalle por pregunta:")
                for i, preg in enumerate(self.preguntas):
                    if i in botones_respuesta:
                        seleccionada = botones_respuesta[i].value
                        correcta = preg['correcta']
                        estado = "✓" if seleccionada == correcta else "✗"
                        print(f"\n  {estado} Pregunta {i+1}: {preg['pregunta']}")
                        print(f"    Tu respuesta: {preg['opciones'][seleccionada]}")
                        if seleccionada != correcta:
                            print(f"    Correcta: {preg['opciones'][correcta]}")
        
        # Crear preguntas
        items_quiz = []
        for i, preg in enumerate(self.preguntas):
            items_quiz.append(widgets.HTML(f"<b>Pregunta {i+1}: {preg['pregunta']}</b>"))
            opciones_widget = widgets.RadioButtons(
                options=preg['opciones'],
                description='',
            )
            botones_respuesta[i] = opciones_widget
            items_quiz.append(opciones_widget)
        
        # Botón para enviar
        btn_enviar = widgets.Button(description='Enviar respuestas', button_style='success')
        btn_enviar.on_click(lambda b: actualizar_resultado(b))
        items_quiz.append(btn_enviar)
        
        items_quiz.append(output)
        
        return widgets.VBox(items_quiz)

quiz = QuizFifo()
print("\n📝 AUTOEVALUACIÓN: Contesta las preguntas y presiona 'Enviar respuestas'\n")
display(quiz.crear_quiz())

# Sección 9: Lecturas y Recursos de Práctica

## Libros Recomendados

1. **"Introduction to Algorithms" (CLRS)**
   - Capítulo 10: Pilas y Colas
   - Análisis riguroso de complejidad

2. **"Data Structures and Algorithms in Python"** - Goodrich, Tamassia
   - Implementaciones detalladas
   - Comparaciones prácticas

3. **"Algorithms Illuminated" - Tim Roughgarden**
   - Explicaciones intuitivas
   - Muchos ejemplos visuales

## Plataformas Interactivas

- **VisuAlgo** (https://visualgo.net): Visualización de BFS y otros algoritmos
- **LeetCode**: Problemas de filas y BFS
  - Easy: Implement Queue using Stacks
  - Medium: Number of Islands (BFS)
  - Medium: Rotting Oranges (BFS)

- **Codeforces**: Concursos de programación
- **HackerRank**: Tutoriales interactivos

## Temas Avanzados

1. **Colas con Prioridad (Priority Queues)**
   - Implementación con heaps
   - Aplicación: Dijkstra's algorithm

2. **Colas Circulares**
   - Optimización de espacio

3. **Colas Concurrentes (Threading)**
   - `queue.Queue` en Python
   - Sincronización en sistemas multi-threaded

4. **Colas Distribuidas**
   - RabbitMQ, Apache Kafka
   - Sistemas de mensajería

# Resumen de la Clase

## Puntos Clave

1. **FIFO** es el principio fundamental de las filas: primero en entrar, primero en salir

2. **Dos implementaciones prácticas:**
   - **FilaDeque**: O(1) amortizado, usa internamente listas doblemente enlazadas
   - **FilaEnlazada**: O(1) garantizado, usa DOS punteros (__frente y __final)

3. **¡NUNCA usar `list.pop(0)`!** Es O(n) porque desplaza todos los elementos

4. **API uniforme:**
   - `enqueue(item)`, `dequeue()`, `front()`, `is_empty()`, `size()`

5. **Aplicaciones clásicas:**
   - Simulaciones (colas de banco, impresoras)
   - BFS - encontrar caminos más cortos
   - Sistemas de tickets y colas de atención

6. **Comparación con Pilas:**
   - Pilas (LIFO): un extremo, backtracking, evaluación de expresiones
   - Filas (FIFO): dos extremos, BFS, equidad en turnos

## Próxima Clase

Colas con Prioridad (Priority Queues) usando heaps - un paso intermedio entre filas simples y algoritmos avanzados.